# 🎙️ Speech Corpus Pipeline: Student 1 (v14 — Modular Runner Notebook)
Covers **Steps 1–4** of the project brief:
1. Video Ingestion & Metadata extraction (`info.json`, Creative Commons verification).
2. Audio conditioning (16 kHz, Mono, 16-bit PCM WAV + FLAC).
3. AST Acoustic Macro-Classification (`speech`, `music`, `mixed`, `silence`) with non-overlapping regions and pause snapping.
4. Continuous 10ms Short-Time Energy curve & Otsu Adaptive Silence Floor.
5. Cut Engine v2 (Global Shortest-Path DP, Whisper Arabic word verification, edge snapping, 100–200ms margins, 8ms Hann fade, whole music regions).
6. 4 Interactive UI Inspectors with 60 FPS moving cursor + 10-minute ground-truth manual check tool.
7. Measured Quality Control & Safe Dataset Splitting.


In [1]:
# ⚙️ Cell 1: Environment Setup & Modular Package Imports
import sys
from pathlib import Path

# Add repository root to Python path so speech_corpus package is loaded cleanly
REPO_PATH = Path('Audio-Datasets-Creation-System')
if REPO_PATH.exists() and str(REPO_PATH.resolve()) not in sys.path:
    sys.path.insert(0, str(REPO_PATH.resolve()))

import numpy as np
import pandas as pd
import torch
from IPython.display import display, Audio, HTML

# 📦 Import directly from the modular speech_corpus package (v2.2)
from speech_corpus.audio import read_wav, write_wav, extract_audio, extract_audio_hq, DEFAULT_SR
from speech_corpus.downloader import download_video, find_cached_video, ingest_local_file, is_creative_commons
from speech_corpus.ast_model import (
    classify_audio, ast_predict_audio_windows, windows_to_regions,
    refine_region_boundaries, get_ast_engine, CONTENT_LABELS,
    validate_cached_regions, save_regions
)
from speech_corpus.segmentation import (
    compute_energy_db, adaptive_silence_floor,
    compute_unified_voting_grid,
    detect_pauses_silero, detect_pauses_energy,
    detect_speech_activity_overlay, compute_instantaneous_energy_vad_overlay,
    log_human_change
)
from speech_corpus.cut_engine_v2 import (
    cut_four_label_segments, cut_speech_and_mixed_segments,
    Cand, PauseCandidate, plan_unified_dp_cuts, greedy_plan,
    apply_hann_fade
)
from speech_corpus.inspector import (
    plot_spectrogram_with_regions,
    launch_spectrogram_region_editor,
    launch_timeline_pointer_inspector,
    launch_vad_overlay_inspector,
    launch_segment_inspector,
    sample_manual_check_window,
    launch_manual_check,
    summarize_manual_check
)
from speech_corpus.segment_editor import launch_segment_editor
from speech_corpus.quality_control import (
    validate_segments_integrity,
    generate_qc_markdown_report,
    video_summary,
    build_corpus_manifest,
    split_corpus
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dev_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Host CPU'
print(f"🚀 Environment Ready! • Device: {device.upper()} ({dev_name})")
print('✅ Modular Speech Corpus Package Loaded (Specification Version 2.2 — Unified Four-Label Voting).')


🚀 Environment Ready! • Device: CUDA (NVIDIA GeForce RTX 3050 Laptop GPU)
✅ Modular Speech Corpus Package Loaded (Specification Version 2.2 — Unified Four-Label Voting).


## ⚙️ Pipeline Configuration


In [2]:
# ⚙️ Cell 2: Configuration Parameters
USE_MOCK_DEMO = False          # Set to True for an instant offline smoke test
YOUTUBE_URL = "https://www.youtube.com/watch?v=gK017cCH36k"  # Video URL (>= 1 hour recommended)
LOCAL_FILE = None              # Optional: path to local .mp4/.wav (e.g. "my_video.mp4")
COOKIES_FILE = None            # Optional: path to cookies.txt if YouTube blocks IP
DATA_ROOT = Path("data")       # Storage directory for all data and segments
ANNOTATOR = "Amr"              # Annotator name for audit trail in changes.csv

SILENCE_THRESHOLD_S = 0.25     # 250ms threshold: minimum pause duration for a 100% safe cut
MAX_HEIGHT = 360               # Resolution cap for fast video download
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Configured: Annotator='{ANNOTATOR}' | Data Root='{DATA_ROOT.resolve()}'")


Configured: Annotator='Amr' | Data Root='C:\Users\Amr\Downloads\Dsp project\data'


## 📥 Steps 1 & 2: Video Download, Audio Conditioning & Metadata (`info.json`)


In [3]:
# 📥 Cell 3: Audio Ingestion & Metadata
if USE_MOCK_DEMO:
    from speech_corpus.labelling import build_mock_audio
    print("⚡ Generating synthetic multi-class test audio...")
    video_id = "mock_demo_audio"
    video_dir = DATA_ROOT / video_id
    video_dir.mkdir(parents=True, exist_ok=True)
    audio_path = video_dir / "audio.wav"
    audio_samples, gt_df = build_mock_audio(seed=42, sr=DEFAULT_SR)
    write_wav(audio_path, audio_samples, sample_rate=DEFAULT_SR)
    info = {
        "video_id": video_id, "source": "mock", "url": None, "title": "Synthetic Audio Demo",
        "channel": "Synthesizer", "duration_s": round(len(audio_samples) / DEFAULT_SR, 2),
        "license": "Creative Commons Attribution", "is_creative_commons": True,
        "downloaded_at": "2026-11-21"
    }
    (video_dir / "info.json").write_text(json.dumps(info, indent=2, ensure_ascii=False), encoding="utf-8")
elif LOCAL_FILE is not None:
    print(f"📂 Ingesting local file: {LOCAL_FILE}")
    video_id = Path(LOCAL_FILE).stem
    video_dir = DATA_ROOT / video_id
    audio_path, info = ingest_local_file(LOCAL_FILE, video_dir)
    audio_samples, sr = read_wav(audio_path, target_sr=DEFAULT_SR)
else:
    cached = find_cached_video(YOUTUBE_URL, search_dirs=[DATA_ROOT])
    if cached is not None:
        audio_path, info = cached
        video_dir = audio_path.parent
        print(f"⚡ Found cached audio for video [{info.get('video_id')}]: {audio_path}")
        audio_samples, sr = read_wav(audio_path, target_sr=DEFAULT_SR)
    else:
        print(f"📥 Downloading and extracting audio from: {YOUTUBE_URL}")
        from speech_corpus.downloader import extract_video_id
        vid = extract_video_id(YOUTUBE_URL)
        video_dir = DATA_ROOT / vid
        audio_path, info = download_video(YOUTUBE_URL, out_dir=video_dir, cookies_file=COOKIES_FILE)
        audio_samples, sr = read_wav(audio_path, target_sr=DEFAULT_SR)

dur_s = len(audio_samples) / DEFAULT_SR
print(f"🎵 Audio Ready: {dur_s:.1f}s ({dur_s/60:.2f} mins) @ {DEFAULT_SR} Hz | Directory: {video_dir}")
print(f"📄 License: '{info.get('license')}' | Creative Commons: {'✅ YES' if info.get('is_creative_commons') else '⚠️ NO'}")


📥 Downloading and extracting audio from: https://www.youtube.com/watch?v=gK017cCH36k
⚠️ License is 'Standard YouTube License', not Creative Commons. Document this in the report.
🎵 Audio Ready: 5740.8s (95.68 mins) @ 16000 Hz | Directory: data\gK017cCH36k
📄 License: 'Standard YouTube License' | Creative Commons: ⚠️ NO


## ⚡ Step 3.0: Millisecond VAD Pauses & Otsu Adaptive Silence Floor
Extracts true acoustic pauses and computes the short-time RMS energy (vectorised cumsum) with an Otsu threshold on the dB histogram. Needed to snap region borders before cutting.


In [4]:
# ⚡ Cell 4: Speech Activity & Adaptive Silence Floor
print("🔍 Extracting VAD speech pauses and vectorised short-time RMS energy...")
vad_timeline_df, pauses_df = detect_speech_activity_overlay(
    audio=audio_samples, sr=DEFAULT_SR,
    silence_threshold_s=SILENCE_THRESHOLD_S,
    min_pause_ms=120, vad_backend="auto"
)

intervals_df, times_arr, energy_db_arr, thresh_db = compute_instantaneous_energy_vad_overlay(
    audio=audio_samples, sr=DEFAULT_SR, vad_timeline_df=vad_timeline_df
)
floor_db, _ = adaptive_silence_floor(energy_db_arr)

sp_dur = vad_timeline_df.loc[vad_timeline_df['state'] == 'SPEECH', 'duration'].sum() if not vad_timeline_df.empty else 0.0
sil_dur = vad_timeline_df.loc[vad_timeline_df['state'] == 'SILENCE', 'duration'].sum() if not vad_timeline_df.empty else 0.0

print(f"• Active Speech Time   : {sp_dur/60:.2f} min ({100*sp_dur/dur_s:.1f}%)")
print(f"• Silence / Pause Time : {sil_dur/60:.2f} min ({100*sil_dur/dur_s:.1f}%)")
print(f"• Adaptive Silence Floor: {floor_db:.1f} dB (Threshold: {thresh_db:.1f} dB)")
print(f"• Pauses Detected      : {len(pauses_df)} pauses (Safe pauses >= 250ms: {sum(pauses_df['is_safe'])})")


🔍 Extracting VAD speech pauses and vectorised short-time RMS energy...
• Active Speech Time   : 63.54 min (66.4%)
• Silence / Pause Time : 32.14 min (33.6%)
• Adaptive Silence Floor: -38.5 dB (Threshold: -25.5 dB)
• Pauses Detected      : 2578 pauses (Safe pauses >= 250ms: 2062)


## 🧠 Step 3: AST Acoustic Classification & Overlap-Free Region Tiling
Predicts multi-class AudioSet labels (Speech, Music, Silence, Mixed), tiles the audio using window centers (0% overlap bug solved), and snaps region edges to the nearest VAD pause.


In [5]:
# 🧠 Cell 5: AST Classification & Non-Overlapping Region Tiling (with v2.2 Cache Validation)
regions_csv = video_dir / 'regions.csv'
windows_csv = video_dir / 'windows_ast.csv'
FORCE_RECLASSIFY = False

# Cache validation against source ID, duration, and version 2.2
is_valid_cache = (
    not FORCE_RECLASSIFY and
    validate_cached_regions(regions_csv, expected_source_id=video_dir.name, expected_duration=dur_s, expected_version='2.2')
)

if is_valid_cache:
    print(f'⚡ Valid cached regions found (v2.2): {regions_csv}')
    regions_df = pd.read_csv(regions_csv)
    windows_df = pd.read_csv(windows_csv) if windows_csv.exists() else None
else:
    print('🧠 Running Audio Spectrogram Transformer (AST) neural inference...')
    regions_df, windows_df = classify_audio(
        y=audio_samples, sr=DEFAULT_SR, win_s=10.0, hop_s=5.0, batch_size=32,
        pauses_df=pauses_df
    )
    save_regions(regions_df, regions_csv, source_id=video_dir.name, duration=dur_s, version='2.2')
    if windows_df is not None and not windows_df.empty:
        windows_df.to_csv(windows_csv, index=False)
    print(f'Saved: {regions_csv} (v2.2 validated cache) and {windows_csv}')

print(f'Identified {len(regions_df)} non-overlapping acoustic regions across {dur_s:.1f}s:')
display(regions_df.head(10))


🧠 Running Audio Spectrogram Transformer (AST) neural inference...
[AST] Loading 'MIT/ast-finetuned-audioset-10-10-0.4593' on CUDA...


Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

[AST] class ids -> speech:[0, 1, 2, 3, 4, 5] music:[27, 137, 138, 266] silence:[500]
Saved: data\gK017cCH36k\regions.csv (v2.2 validated cache) and data\gK017cCH36k\windows_ast.csv
Identified 15 non-overlapping acoustic regions across 5740.8s:


,region_id,start,end,duration,label,confidence,urgency,review_priority,status,refined
0,R0001,0.00,164.05,164.05,speech,0.775,0.244,LOW,unreviewed,True
1,R0002,164.05,173.05,9.00,silence,0.900,0.791,URGENT,unreviewed,True
2,R0003,173.05,2273.05,2100.00,speech,0.941,0.066,LOW,unreviewed,True
3,R0004,2273.05,2286.25,13.20,silence,0.900,0.495,MEDIUM,unreviewed,True
4,R0005,2286.25,2665.45,379.20,speech,0.938,0.064,LOW,unreviewed,True
5,R0006,2665.45,2683.55,18.10,silence,0.900,0.587,MEDIUM,unreviewed,True
6,R0007,2683.55,3935.85,1252.30,speech,0.948,0.057,LOW,unreviewed,True
7,R0008,3935.85,3947.15,11.30,silence,0.900,0.927,URGENT,unreviewed,True
8,R0009,3947.15,3978.85,31.70,speech,0.938,0.066,LOW,unreviewed,True
9,R0010,3978.85,4024.15,45.30,silence,0.900,0.187,LOW,unreviewed,True


### 🔍 Interactive Inspector 1: Spectrogram & Macro-Region Editor
Inspect linear spectrogram (0–8 kHz), listen, move start/end boundaries, relabel, split, merge, or delete regions. Every change updates `regions.csv` and is logged to `changes.csv` with annotator name.


In [6]:
# 🔍 Cell 6: [Inspector 1] Spectrogram + Region Editor
launch_spectrogram_region_editor(
    audio=audio_samples,
    regions_df=regions_df,
    video_dir=video_dir,
    sr=DEFAULT_SR,
    user=ANNOTATOR
)


### 📍 Interactive Inspector 2: 60 FPS Real-Time Moving Pointer
Synchronized moving red cursor rendered inside an HTML5 Canvas + Web Audio API player. Scrub through audio and watch the instant voice activity classification update at 60 FPS.


In [7]:
# 📍 Cell 7: [Inspector 2] 60 FPS Moving Pointer Across Waveform & Energy Curve
launch_timeline_pointer_inspector(
    audio=audio_samples,
    pauses_df=pauses_df,
    vad_timeline_df=vad_timeline_df,
    sr=DEFAULT_SR
)


### 🔍 Interactive Inspector 3: No-Speech Pauses & Safe-Zone Visualizer
Inspect individual pauses, measure their millisecond duration, and verify whether they meet the >= 250ms safety clearance threshold.


In [8]:
# 🔍 Cell 8: [Inspector 3] Pauses and Safe Zones Inspector
launch_vad_overlay_inspector(
    audio=audio_samples,
    pauses_df=pauses_df,
    vad_timeline_df=vad_timeline_df,
    sr=DEFAULT_SR
)


## ✂️ Step 4: Cut Engine v2 (Global DP Optimizer, Word Verification & Whole Music)
Executes global shortest-path DP over candidate pauses. Snaps stream edges to pauses, merges short streams (< 5s), checks Arabic stop consonants and word boundaries via faster-whisper (`language="ar"`), preserves music regions as whole (`music_whole`), enforces 100–200ms margins with 8ms Hann fade, and measures greedy ablation side-by-side.


In [9]:
# ✂️ Cell 9: Execute Cut Engine v2 (Unified Four-Label Global DP)
USE_WORD_VERIFICATION = False   # Set to True for faster-whisper Arabic word boundary checks
WORD_MODEL_SIZE = 'small'       # 'small' or 'tiny' for word boundary verification

segments_df = cut_speech_and_mixed_segments(
    audio=audio_samples,
    regions_df=regions_df,
    out_dir=video_dir,
    sr=DEFAULT_SR,
    pauses=pauses_df,
    use_word_verification=USE_WORD_VERIFICATION,
    word_model_size=WORD_MODEL_SIZE,
    run_ablation=True
)

print(f'Generated {len(segments_df)} segments in {video_dir / "segments"}:')
cols_show = [c for c in ['segment_id', 'start', 'end', 'duration', 'primary_label', 'label_composition', 'cut_tier', 'voting_agreement', 'decision_class', 'status'] if c in segments_df.columns]
display(segments_df[cols_show].head(10))


ℹ️ Loaded regions.csv (includes macro-regions and any human edits).
🎯 CUT ENGINE V2 — UNIFIED FOUR-LABEL PIPELINE [PASSED]
• Total segments generated     : 435 (total audio: 95.68 min)
• Primary window (5–15 s)      : 435 / 435 (100.0%)
• Fallback window (3–18 s)     : 0 / 435
• Margin 100–200 ms satisfied  : 100.0%
• Whisper word checks          : verified 0, split risks 0
• Review workload              : 0 / 435 (0.0%) [target 5–15%]
• Mandatory QC gates passed    : 10 / 10
• Output manifest              : data\gK017cCH36k\segments.csv
Generated 435 segments in data\gK017cCH36k\segments:


,segment_id,start,end,duration,primary_label,label_composition,cut_tier,voting_agreement,decision_class,status
0,1,0.00,14.15,14.15,speech,speech:14.15,HIGH,2/2,STRONG,PASSED
1,2,14.15,27.80,13.65,speech,speech:13.65,HIGH,2/2,STRONG,PASSED
2,3,27.80,38.55,10.75,speech,speech:10.75,HIGH,2/2,STRONG,PASSED
3,4,38.55,53.15,14.60,speech,speech:14.60,HIGH,2/2,STRONG,PASSED
4,5,53.15,67.15,14.00,speech,speech:14.00,HIGH,2/2,STRONG,PASSED
5,6,67.15,81.45,14.30,speech,speech:14.30,HIGH,2/2,STRONG,PASSED
6,7,81.45,96.15,14.70,speech,speech:14.70,HIGH,2/2,STRONG,PASSED
7,8,96.15,110.75,14.60,speech,speech:14.60,HIGH,2/2,STRONG,PASSED
8,9,110.75,122.85,12.10,speech,speech:12.10,HIGH,2/2,STRONG,PASSED
9,10,122.85,137.40,14.55,speech,speech:14.55,HIGH,2/2,STRONG,PASSED


### 🔍 Interactive Inspector 4: Cut Segments & Human Review Queue
Listen to individual cut segments, inspect certainty tiers (HIGH, MEDIUM, LOW), relabel if needed, and mark segments reviewed with audit logging.


In [10]:
# 🔍 Cell 10: [Inspector 4] Segment Review Queue & Audit Logging
launch_segment_inspector(
    audio=audio_samples,
    segments_df=segments_df,
    video_dir=video_dir,
    sr=DEFAULT_SR,
    user=ANNOTATOR
)


### ✂️ Segment Editor: Move, Split, Merge, Delete & Add Segments
Manually adjust segment start/end boundaries, split or merge adjacent segments, delete unwanted segments, or add new segments. Every change updates `segments.csv`, `segments/*.wav`, and `changes.csv`.


In [ ]:
# ✂️ Cell 10b: Interactive Segment Editor (Move / Split / Merge / Delete / Add)
seg_editor_state = launch_segment_editor(
    audio=audio_samples,
    segments_df=segments_df,
    video_dir=video_dir,
    sr=DEFAULT_SR,
    regions_df=regions_df,
    user=ANNOTATOR,
)
# After finishing UI edits, run: segments_df = seg_editor_state["df"]


## 📋 Step 5: Measured Quality Control (QC) & Ablation Report
Evaluates real, countable integrity checks (no overlapping segments, duration bounds, margins, audio file validity) and compiles the side-by-side measured ablation table (Greedy Baseline vs. Global DP Optimizer).


In [11]:
# 📋 Cell 11: Quality Control Integrity (10 Mandatory Gates) & Ablation Report
qc_results = validate_segments_integrity(segments_df, video_dir=video_dir, total_dur=dur_s, sr=DEFAULT_SR)
qc_md = generate_qc_markdown_report(
    segments_df, report_path=video_dir / 'qc_report.md',
    video_dir=video_dir, qc=qc_results
)

passed_cnt = qc_results['summary']['passed_gates']
total_cnt = qc_results['summary']['total_gates']
print(f"QC Status: {qc_results['status']} ({passed_cnt}/{total_cnt} mandatory gates passed)")
for g in qc_results['gates']:
    icon = '✅ PASS' if g['passed'] else '❌ FAIL'
    print(f"  {icon} Gate {g['gate_id']}: {g['name']} — {g['detail']}")
if qc_results['errors']:
    print('❌ Errors:', qc_results['errors'])
if qc_results['warnings']:
    print('⚠️ Warnings:', qc_results['warnings'])

print()
print('QC Report Preview:')
print('-' * 65)
print(qc_md)


QC Status: PASSED (10/10 mandatory gates passed)
  ✅ PASS Gate 1: Full Source Coverage & No Overlaps — 0 overlaps, 0 unexplained gaps
  ✅ PASS Gate 2: Valid Four-Label Processing — Labels present: ['silence', 'speech'], Invalid: []
  ✅ PASS Gate 3: Duration Window Compliance (5-15s primary >=85%, 4-17s fallback flagged) — Primary window: 100.0%, Undersized (<4s): 0, Oversized (>17s): 0, Unflagged fallbacks: 0
  ✅ PASS Gate 4: Continuous Silence (>15s) Direct Voting Grid Validation — Silence cuts audit: 11 supported, 0 fallback/review, 0 invalid
  ✅ PASS Gate 5: Midpoint Correctness Verification (midpoint_error_ms <= 2.0 ms) — Max midpoint error: 0.00 ms, Mean: 0.00 ms (0 cuts exceeded 2.0 ms tolerance) | Edge margin compliance: 100.0%
  ✅ PASS Gate 6: Word Safety Verification & Risk Flagging — 0 unflagged word split risks
  ✅ PASS Gate 7: Label Composition & Overlap Tracking — Composition columns present and populated
  ✅ PASS Gate 8: Audio Files Existence, Readability & Duration Match

## 🧪 Step 5.5: 10-Minute Ground-Truth Manual Check (Required by Project Brief)
Samples a random contiguous 10-minute window of segments for manual verification. Listen, tick any errors (wrong label / cut inside word), save, and calculate bad cut calibration rates.


In [12]:
# 🧪 Cell 12: Ground-Truth Manual Quality Check UI (Full Video & Review Queue)
import importlib
import speech_corpus.inspector
importlib.reload(speech_corpus.inspector)
from speech_corpus.inspector import launch_manual_check, sample_manual_check_window

# 1. Initialize full-video manual check (all segments, not just 10 minutes)
check_sample_df = sample_manual_check_window(segments_df, video_dir, minutes=None)

# 2. Launch interactive review (2 options: Flagged Only vs Full Video | System Label Display | Labelling & Segmentation review)
launch_manual_check(audio_samples, video_dir, segments_df=segments_df, sr=DEFAULT_SR)

Manual-check full video: 435 segments, 95.7 min -> manual_check.csv


In [13]:
# 📊 Cell 13: Summarize Manual Check Results
# Run this cell after you have finished ticking the segments in the checklist above
manual_stats = summarize_manual_check(video_dir)


Checked 0/435 segments (0.0 min)


## 📂 Step 6: Summary Table, Corpus Manifest & Zero-Leakage Dataset Splitting
Generates `video_summary.csv` with durations of speech, music, mixed, and silence. Compiles all videos' segments into `corpus_manifest.csv` and performs safe cross-video (or time-block) dataset splitting (70% Train, 15% Val, 15% Test).


In [14]:
# 📂 Cell 14: Summary, Manifest & Zero-Leakage Splitting
summary_df = video_summary(
    regions_df=pd.read_csv(video_dir / 'regions.csv'),
    segments_df=segments_df,
    info=info,
    video_dir=video_dir
)
print('📊 Video Duration & Labelling Summary:')
display(summary_df)

# Global corpus manifest across all processed videos
manifest_df = build_corpus_manifest(DATA_ROOT, DATA_ROOT / 'corpus_manifest.csv')
print(f"📄 Corpus Manifest: {len(manifest_df)} total segments from {manifest_df['video_id'].nunique()} video(s)")

# Dataset splitting (strictly source-level, zero leakage)
splits = split_corpus(manifest_df, out_dir=DATA_ROOT / 'splits', ratios=(0.70, 0.15, 0.15))
if splits.get('status') == 'SUCCESS':
    print(f"✅ Splits Saved in {DATA_ROOT / 'splits'}: Train={len(splits['splits']['train'])}, Val={len(splits['splits']['val'])}, Test={len(splits['splits']['test'])}")
else:
    print(f"ℹ️ Split Result: {splits.get('message', 'Infeasible (fewer than 3 sources)')}")


📊 Video Duration & Labelling Summary:


,video,duration_min,speech_min,music_min,mixed_min,silence_min,segments,mean_seg_s,review_pct,license,url
0,DSP - Elective 2 || Lecture 1,95.68,93.62,0.0,0.0,2.06,435,13.2,0.0,Standard YouTube License,https://www.youtube.com/watch?v=gK017cCH36k


📄 Corpus Manifest: 881 total segments from 7 video(s)
🎯 LEAKAGE-SAFE SOURCE-LEVEL SPLIT COMPLETED
• Total sources: 7 | Total audio: 173.77 min
• Split 'train': 5 sources, 451 segments, 99.72 min (57.4%)
• Split 'val': 1 sources, 360 segments, 59.84 min (34.4%)
• Split 'test': 1 sources, 70 segments, 14.21 min (8.2%)
✅ Splits Saved in data\splits: Train=451, Val=360, Test=70


## 💾 Step 7: Packaging & Downstream Handoff
Creates a unified zip archive containing `segments.csv`, audio WAV files, metadata, QC reports, and splits, ready for handoff to Student 2 (Denoising / Noise Filtering) and Student 3 (Web App / UI).


In [15]:
# 💾 Cell 15: Create Handoff Zip Archive
import zipfile

zip_filename = f"{video_dir.name}_student1_handoff.zip"
with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as z:
    for f in video_dir.rglob("*"):
        if f.is_file() and not f.name.endswith(".zip"):
            z.write(f, arcname=f.relative_to(video_dir.parent))

print(f"🎉 Pipeline Complete! Handoff archive created: {Path(zip_filename).resolve()} ({Path(zip_filename).stat().st_size / (1024*1024):.2f} MB)")
print("🚀 Ready for handoff to Student 2 and Student 3!")


🎉 Pipeline Complete! Handoff archive created: C:\Users\Amr\Downloads\Dsp project\gK017cCH36k_student1_handoff.zip (319.99 MB)
🚀 Ready for handoff to Student 2 and Student 3!
